# Graphs & referrals

Generated from the FeatureMesh docs tutorial. This variant targets the FeatureMesh demos Jupyter environment.


## Set up FeatureMesh

Load the Jupyter magic and create a local `BatchClient` for the demos Jupyter environment. Run these cells once before the tutorial.


In [1]:
%load_ext featuremesh


In [2]:
from IPython.display import display
from featuremesh import BatchClient, set_default

client = BatchClient()
set_default("client", client)
print("FeatureMesh BatchClient ready (local DuckDB)")


FeatureMesh BatchClient ready (local DuckDB)


Walk a customer referral tree with `RECURSE()`: depth from organic roots, revenue for each subtree, a NULL-segment revenue audit, and connected components from a short edge list — no separate graph database required.

This advanced tutorial assumes entity bindings and `RELATED()` from [E-commerce](https://featuremesh.com/docs/tutorials/analytics/ecomm). `RECURSE()` is the new concept: seed a row, follow an edge expression, and stop at `MAX_LEVEL`. Run Data and Model before the graph queries.


## Data

Two organic roots. **Alpha** refers Bravo and Charlie; Bravo refers Delta; Delta refers Echo (**segment NULL**, so segment rollups miss that revenue). **Golf** refers Hotel. The connected-components query inlines a tiny undirected edge list.

```
Alpha (organic)
├── Bravo
│   └── Delta
│       └── Echo   ← segment NULL
└── Charlie
Golf (organic)
└── Hotel
```


In [3]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN FM.GRAPH UP TO LEVEL 9;


*** Warning(s) (1) ***

  1. [DROP-FEATURES-EXPRESSION-EMPTY] No features found in expression: *(FEATURES IF EXISTS IN FM.GRAPH UP TO LEVEL 9) (acknowledge with ACK-HWQ3)


In [4]:
%%featureql --client client

/* SQL */
CREATE SCHEMA IF NOT EXISTS tutorial_graph;
--
DROP TABLE IF EXISTS tutorial_graph.revenue;
--
DROP TABLE IF EXISTS tutorial_graph.customers;
--
CREATE TABLE tutorial_graph.customers (
  id BIGINT,
  name VARCHAR,
  segment VARCHAR,
  referred_by BIGINT
);
--
INSERT INTO tutorial_graph.customers VALUES
  (1, 'Alpha',   'enterprise', NULL),
  (2, 'Bravo',   'smb',        1),
  (3, 'Charlie', 'smb',        1),
  (4, 'Delta',   'mid_market', 2),
  (5, 'Echo',    NULL,         4),
  (6, 'Golf',    'enterprise', NULL),
  (7, 'Hotel',   'smb',        6);
--
CREATE TABLE tutorial_graph.revenue (
  customer_id BIGINT,
  amount BIGINT
);
--
INSERT INTO tutorial_graph.revenue VALUES
  (1, 5000),
  (2, 3000),
  (3, 4500),
  (4, 2000),
  (5,  800),
  (6, 6000),
  (7, 3500);
--
SELECT CAST(COUNT(*) AS INTEGER) AS cnt FROM tutorial_graph.customers;


,cnt
0,7


## Model

Customers carry `referred_by` (NULL = organic). Revenue is one amount per customer.


In [5]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.GRAPH AS
SELECT
    customers := ENTITY(),
    customer_id := INPUT(BIGINT#customers)
;


,feature_name,status,message
0,FM.GRAPH.CUSTOMERS,CREATED,Feature created as not exists
1,FM.GRAPH.CUSTOMER_ID,CREATED,Feature created as not exists


In [6]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.GRAPH AS
SELECT
    tables.customers := EXTERNAL_COLUMNS(
        id BIGINT#customers BIND TO customer_id,
        name VARCHAR,
        segment VARCHAR,
        referred_by BIGINT
        FROM TABLE(tutorial_graph.customers)
    ),
    customer_name := tables.customers[name],
    ref_by := tables.customers[referred_by],
    segment := tables.customers[segment],
    tables.customer_revenue := EXTERNAL_COLUMNS(
        customer_id BIGINT#customers BIND TO customer_id,
        amount BIGINT
        FROM TABLE(tutorial_graph.revenue)
    ),
    rev_amt := tables.customer_revenue[amount]
;


,feature_name,status,message
0,FM.GRAPH.TABLES.CUSTOMERS,CREATED,Feature created as not exists
1,FM.GRAPH.CUSTOMER_NAME,CREATED,Feature created as not exists
2,FM.GRAPH.REF_BY,CREATED,Feature created as not exists
3,FM.GRAPH.SEGMENT,CREATED,Feature created as not exists
4,FM.GRAPH.TABLES.CUSTOMER_REVENUE,CREATED,Feature created as not exists
5,FM.GRAPH.REV_AMT,CREATED,Feature created as not exists


## Referral depth

From each customer, walk parent links until the organic root. Depth = hops from organic (organic itself is **0**).


In [7]:
%%featureql --client client

WITH
    CUST_ID := TABLES.CUSTOMERS[id],
    WALK := ROW(CUSTOMER_ID AS start).RECURSE(
        SELECT step[cust_id] AS node_id
        VIA start BIND TO cust_id
        FOLLOW ref_by
        MAX_LEVEL 10
    ),
    REF_DEPTH := COALESCE(WALK.TRANSFORM(SELECT MAX(level)).UNWRAP_ONE(), 0) - 1,
SELECT
    CUSTOMER_NAME,
    REF_DEPTH
FROM FM.GRAPH
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[1, 2, 3, 4, 5, 6, 7])
ORDER BY CUSTOMER_NAME;


,FM.GRAPH.CUSTOMER_NAME,REF_DEPTH
0,Alpha,0
1,Bravo,1
2,Charlie,1
3,Delta,2
4,Echo,3
5,Golf,0
6,Hotel,1


Alpha/Golf **0**, Bravo/Charlie/Hotel **1**, Delta **2**, Echo **3**.

## Subtree revenue

Same walk, but keep the **organic root** for every node, then `SUM` revenue by root (includes the root’s own revenue).


In [8]:
%%featureql --client client

WITH
    cust_id := tables.customers[id],
    walk := ROW(customer_id AS start).RECURSE(
        SELECT node_id := step[cust_id]
        VIA start BIND TO cust_id
        FOLLOW ref_by
        MAX_LEVEL 10
    ),
    organic_root := COALESCE(
        walk.TRANSFORM(SELECT NODE_ID ORDER BY LEVEL DESC LIMIT 1).UNWRAP_ONE(),
        customer_id
    ),
    tree_rev := SUM(rev_amt) GROUP BY organic_root
SELECT
    root_id := organic_root,
    tree_rev
FROM FM.GRAPH
FOR
    customer_id := BIND_VALUES(ARRAY(1, 2, 3, 4, 5, 6, 7))
ORDER BY root_id
;


,ROOT_ID,TREE_REV
0,1,15300
1,6,9500


Alpha tree **15300**. Golf tree **9500**.

## NULL segment audit

Total revenue vs sum of revenue where `segment IS NOT NULL`. The gap is Echo’s **800**.


In [9]:
%%featureql --client client

WITH
    total_rev := SUM(rev_amt),
    parts_sum := SUM(rev_amt) FILTER (WHERE segment IS NOT NULL),
    delta := total_rev - parts_sum
SELECT
    total_rev,
    parts_sum,
    delta
FROM FM.GRAPH
FOR
    customer_id := BIND_VALUES(ARRAY(1, 2, 3, 4, 5, 6, 7))
;


,TOTAL_REV,PARTS_SUM,DELTA
0,24800,24000,800


**24800 − 24000 = 800**. Group-by dimensions silently drop NULLs — check the delta.

## Connected components

Customers that share an edge are in the same component (transitive). Seed each node, `RECURSE` across edges, take `MIN` reachable id as the component label.


In [10]:
%%featureql --client client

WITH
    EDGES_E := ENTITY(),
    EDGE_ID := INPUT(BIGINT#EDGES_E),
    EDGE_TABLE := INLINE_COLUMNS(
        edge_id BIGINT#EDGES_E BIND TO EDGE_ID,
        edge_src BIGINT#CUSTOMERS,
        edge_dst BIGINT#CUSTOMERS
        FROM CSV(
            edge_id,edge_src,edge_dst
            1,1,2
            2,2,1
            3,2,4
            4,4,2
            5,6,7
            6,7,6
        )
    ),
    EDGE_SRC := EDGE_TABLE[edge_src],
    EDGE_DST := EDGE_TABLE[edge_dst],
    REACH := ROW(CUSTOMER_ID AS start).RECURSE(
        SELECT step[edge_dst] AS peer
        VIA start BIND TO edge_src
        FOLLOW edge_dst
        MAX_LEVEL 10
    ),
    COMPONENT_ID := REACH.CARRY(CUSTOMER_ID AS seed).TRANSFORM(
        SELECT MIN(COALESCE(peer, seed)) AS component_id
    ).UNWRAP_ONE(),
SELECT
    CUSTOMER_NAME,
    COMPONENT_ID
FROM FM.GRAPH
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[1, 2, 3, 4, 5, 6, 7]),
    NESTED EDGE_ID := BIND_VALUES(ARRAY[1, 2, 3, 4, 5, 6])
ORDER BY CUSTOMER_NAME;


,FM.GRAPH.CUSTOMER_NAME,COMPONENT_ID
0,Alpha,1
1,Bravo,1
2,Charlie,3
3,Delta,1
4,Echo,5
5,Golf,6
6,Hotel,6


Alpha/Bravo/Delta → **1**. Charlie → **3**. Golf/Hotel → **6**. Echo has no edges → **5**.

## What's next

- [Analytics overview](https://featuremesh.com/docs/tutorials/analytics/overview) — concept map for this series
- [OBT modeling](https://featuremesh.com/docs/tutorials/analytics/obt) — array-of-rows, nested argmax, profile `ROW`s
- [Temporal & experiments](https://featuremesh.com/docs/tutorials/analytics/temporal) — SCD as-of and bi-temporal cutoffs
- [Product analytics](https://featuremesh.com/docs/tutorials/analytics/product) — funnels on event logs
- [Financial consolidation](https://featuremesh.com/docs/tutorials/analytics/finance) — hierarchy-style rollups without graphs


---

Source tutorial: [/docs/tutorials/analytics/graph](https://featuremesh.com/docs/tutorials/analytics/graph)
